In [83]:
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score


projects = pd.read_csv("train.csv")

In [10]:
projects.columns

Index(['project_id', 'num_commits', 'active_commit_days', 'project_span_days',
       'commit_cadence', 'pct_commits_weekend', 'pct_commits_late_night',
       'lines_added_total', 'lines_deleted_total', 'churn_ratio',
       'median_lines_per_commit', 'max_lines_in_one_commit',
       'pct_code_in_first_commit', 'num_commit_authors',
       'avg_commit_msg_length', 'pct_generic_commit_msgs', 'num_branches',
       'num_merge_commits', 'lines_of_code', 'num_source_files',
       'num_directories', 'max_dir_depth', 'num_languages', 'primary_language',
       'pct_primary_language', 'num_dependencies', 'framework',
       'uses_game_engine', 'has_tests', 'num_test_files', 'has_ci_config',
       'has_dockerfile', 'has_backend', 'has_database', 'avg_file_length',
       'comment_line_ratio', 'num_image_files', 'num_audio_files',
       'num_3d_model_files', 'num_font_files', 'asset_size_mb',
       'has_custom_art', 'readme_word_count', 'readme_num_headings',
       'readme_num_images', '

In [81]:
y = projects.hours_spent
# Numerical features
project_features = ['num_commits', 'active_commit_days', 'project_span_days',
       'pct_commits_weekend', 'pct_commits_late_night', 'lines_added_total',
       'lines_deleted_total', 'churn_ratio', 'median_lines_per_commit',
       'max_lines_in_one_commit', 'pct_code_in_first_commit',
       'num_commit_authors', 'avg_commit_msg_length',
       'pct_generic_commit_msgs', 'num_branches', 'num_merge_commits',
       'lines_of_code', 'num_source_files', 'num_directories', 'max_dir_depth',
       'num_languages', 'pct_primary_language', 'num_dependencies',
       'num_test_files', 'avg_file_length', 'comment_line_ratio',
       'num_image_files', 'num_audio_files', 'num_3d_model_files',
       'num_font_files', 'asset_size_mb', 'readme_word_count',
       'readme_num_headings', 'readme_num_images', 'readme_num_code_blocks',
       'page_weight_kb', 'num_js_files_loaded', 'num_internal_pages',
       'load_time_ms', 'description_word_count', 'stars', 'forks',
       'num_issues', 'num_pull_requests', 'num_releases',
       'has_custom_art', 'is_mobile_responsive', 'is_multiplayer', 
       'mentions_ai_api', 'is_clone_or_tutorial']

print(len(project_features))

object_cols = ['has_custom_art', 'is_mobile_responsive', 
               'is_multiplayer', 'mentions_ai_api', 
               'is_clone_or_tutorial']

drop = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
X = projects.drop(columns=drop)

cat_cols = X.select_dtypes(include=["str", "object"]).columns
X[cat_cols] = X[cat_cols].astype("category")

X.head()

50


,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,median_lines_per_commit,...,mentions_ai_api,is_clone_or_tutorial,stars,forks,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years
0,44,15,56,daily,0.2045,0.4545,8500,2900,0.3438,128.5,...,False,False,2,0,0,0,0,False,0,1-2
1,9,3,76,sporadic,0.8889,0.0000,500,100,0.2484,32.0,...,False,False,1,0,0,0,0,False,0,3-5
2,12,1,0,frequent,1.0000,0.0000,5900,100,0.0247,64.5,...,False,False,0,0,0,0,0,False,0,3-5
3,4,1,0,frequent,0.0000,0.2500,6000,0,0.0000,1904.0,...,False,False,0,0,0,0,0,False,0,<1
4,7,3,64,sporadic,0.8571,0.0000,800,0,0.0419,198.0,...,False,False,0,0,0,0,0,False,0,1-2


In [99]:
project_model = HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0)

# train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)
# label_train_X = train_X.copy()
# label_val_X = val_X.copy()

# OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(train_X[object_cols]))
# OH_cols_valid = pd.DataFrame(OH_encoder.transform(val_X[object_cols]))

# OH_cols_train.index = train_X.index
# OH_cols_valid.index = val_X.index

# num_X_train = train_X.drop(object_cols, axis=1)
# num_X_valid = val_X.drop(object_cols, axis=1)

# OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
# OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# OH_X_train.columns = OH_X_train.columns.astype(str)
# OH_X_valid.columns = OH_X_valid.columns.astype(str)

# project_model.fit(OH_X_train, np.log1p(train_y))

# print("Predictions for test projects")
# print(project_model.predict(OH_X_valid))
# print("True log hours")
# print(np.log1p(val_y))

# root_mean_squared_log_error(val_y, np.expm1(project_model.predict(OH_X_valid)))
print("RMSLE")
scores = cross_val_score(project_model, X, np.log1p(y), cv=5,
                         scoring="neg_root_mean_squared_error")
print(-scores.mean(), scores.std())

RMSLE
0.672791160858701 0.013860533079410672


In [ ]:
X['commits_per_day'] = X['num_commits'] / X['active_commit_days']
X['total_changed'] = X['lines_added_total'] / X['lines_deleted_total'].replace(0, np.nan)
X['active_day_ratio'] = X['active_commit_days'] / X['project_span_days'].replace(0, np.nan)

In [102]:
# Fine tuning the params for best score

project_model = HistGradientBoostingRegressor(
    categorical_features="from_dtype", 
    learning_rate=0.03, max_iter=10000,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=100, random_state=0)
scores = cross_val_score(project_model, X, np.log1p(y), cv=5,
                        scoring="neg_root_mean_squared_error")
print(-scores.mean(), scores.std())

/home/gauth/venvs/hourdetective/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/home/gauth/venvs/hourdetective/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/home/gauth/venvs/hourdetective/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/home/gauth/venvs/hourdetective/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/home/gauth/venvs/hourdetective/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/home/gauth/venvs/hourdetective/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encou

0.6678228865677351 0.014485555185041346


,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,median_lines_per_commit,...,mentions_ai_api,is_clone_or_tutorial,stars,forks,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years
0,44,15,56,daily,0.2045,0.4545,8500,2900,0.3438,128.5,...,False,False,2,0,0,0,0,False,0,1-2
1,9,3,76,sporadic,0.8889,0.0000,500,100,0.2484,32.0,...,False,False,1,0,0,0,0,False,0,3-5
2,12,1,0,frequent,1.0000,0.0000,5900,100,0.0247,64.5,...,False,False,0,0,0,0,0,False,0,3-5
3,4,1,0,frequent,0.0000,0.2500,6000,0,0.0000,1904.0,...,False,False,0,0,0,0,0,False,0,<1
4,7,3,64,sporadic,0.8571,0.0000,800,0,0.0419,198.0,...,False,False,0,0,0,0,0,False,0,1-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4495,20,2,0,frequent,1.0000,0.0000,1200,100,0.0480,2.0,...,False,False,0,0,0,0,0,False,0,1-2
4496,4,2,36,sporadic,0.0000,0.2500,400,0,0.0049,7.0,...,False,False,0,0,0,0,0,False,0,<1
4497,9,6,100,sporadic,0.5556,0.3333,2000,200,0.1010,90.0,...,False,False,1,0,0,0,1,False,6+,3-5
4498,7,5,19,daily,0.5714,0.0000,181600,0,0.0000,26626.0,...,True,False,0,0,0,0,0,False,0,<1
